# Part 3: KV Cache (Problems 017–024)

Without a KV cache, generating each new token requires recomputing attention over **all previous tokens**. This is O(n²) work per generation step.

A **KV cache** stores the key and value tensors from previous forward passes so we only need to compute attention for the new token against the cached keys/values.

```
Without KV cache:  step 1 → compute [t1]             1 matmul
                   step 2 → compute [t1, t2]          4 matmuls
                   step 3 → compute [t1, t2, t3]      9 matmuls
                   ...

With KV cache:     step 1 → compute [t1], store K1,V1     1 matmul
                   step 2 → compute [t2] + attend K1,V1   2 matmuls
                   step 3 → compute [t3] + attend K1..2   2 matmuls
                   ...
```

## Cell 1: Show the cost of recomputation without cache

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on sys.path so solutions/ is importable
project_root = Path('__file__').parent.parent if '__file__' in dir() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Also try the current directory's parent
for p in [Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'solutions').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break


In [ ]:
import importlib
import time
import torch

try:
    _m = importlib.import_module("solutions.017_understand_kv_recompute_cost")
    compute_attention_cost = getattr(_m, "compute_attention_cost", None) or getattr(_m, "kv_recompute_cost", None)
    if compute_attention_cost is None:
        # Try to get any callable that takes seq_len
        funcs = [v for v in vars(_m).values() if callable(v) and not v.__name__.startswith('_')]
        compute_attention_cost = funcs[0] if funcs else None
except Exception:
    print("Solve problem 017 first:")
    print("  cp problems/017_understand_kv_recompute_cost.py solutions/017_understand_kv_recompute_cost.py")
    compute_attention_cost = None

# Simulate the cost analytically
print("Attention FLOPs (proportional to seq_len^2) at each generation step:")
print()
print(f"  {'Step':>6}  {'Without Cache':>20}  {'With Cache':>14}  {'Speedup':>10}")
print(f"  {'-'*6}  {'-'*20}  {'-'*14}  {'-'*10}")

for step in [1, 5, 10, 20, 50, 100, 200]:
    cost_no_cache = step * step   # O(n^2) per step, accumulated
    cost_with_cache = step        # O(n) per step — only attend to prev tokens
    total_no_cache = sum(i*i for i in range(1, step+1))
    total_with_cache = sum(i for i in range(1, step+1))
    speedup = total_no_cache / total_with_cache
    print(f"  {step:>6}  {total_no_cache:>20,}  {total_with_cache:>14,}  {speedup:>9.1f}x")

print()
print("At 200 steps, the KV cache reduces total FLOPs by ~133x!")

## Cell 2: Allocate KV cache buffers, show shapes

In [ ]:
import importlib
import torch

try:
    _m = importlib.import_module("solutions.018_allocate_kv_cache_buffers")
    allocate_kv_cache_buffers = _m.allocate_kv_cache_buffers
except Exception:
    print("Solve problem 018 first:")
    print("  cp problems/018_allocate_kv_cache_buffers.py solutions/018_allocate_kv_cache_buffers.py")
    allocate_kv_cache_buffers = None

# Model configuration
config = {
    "n_layers": 6,
    "n_heads": 8,
    "d_head": 64,  # d_model / n_heads
    "max_seq_len": 512,
    "max_batch_size": 4,
}

print("Model config:")
for k, v in config.items():
    print(f"  {k}: {v}")

if allocate_kv_cache_buffers is not None:
    cache = allocate_kv_cache_buffers(**config)
    print()
    print("KV cache structure:")
    print(f"  Type: {type(cache).__name__}")
    if isinstance(cache, (list, tuple)):
        print(f"  Number of layers: {len(cache)}")
        if len(cache) > 0:
            layer0 = cache[0]
            if isinstance(layer0, (list, tuple)) and len(layer0) == 2:
                k0, v0 = layer0
                print(f"  Layer 0 K shape: {tuple(k0.shape)}")
                print(f"  Layer 0 V shape: {tuple(v0.shape)}")
    total_bytes = 0
    for layer in cache:
        for tensor in (layer if isinstance(layer, (list, tuple)) else [layer]):
            if hasattr(tensor, 'numel'):
                total_bytes += tensor.numel() * tensor.element_size()
    print()
    print(f"  Total KV cache memory: {total_bytes / 1024**2:.1f} MB")
    expected = 2 * config["n_layers"] * config["max_batch_size"] * config["max_seq_len"] * config["n_heads"] * config["d_head"] * 4
    print(f"  Expected (float32):    {expected / 1024**2:.1f} MB")

## Cell 3: Run prefill phase on a prompt, inspect the cache

In [ ]:
import importlib
import torch

try:
    _m = importlib.import_module("solutions.022_prefill_phase")
    prefill_phase = _m.prefill_phase
except Exception:
    print("Solve problem 022 first:")
    print("  cp problems/022_prefill_phase.py solutions/022_prefill_phase.py")
    prefill_phase = None

try:
    _m = importlib.import_module("solutions.018_allocate_kv_cache_buffers")
    allocate_kv_cache_buffers = _m.allocate_kv_cache_buffers
except Exception:
    allocate_kv_cache_buffers = None

if prefill_phase is not None and allocate_kv_cache_buffers is not None:
    n_layers, n_heads, d_head = 2, 4, 16
    cache = allocate_kv_cache_buffers(
        n_layers=n_layers, n_heads=n_heads, d_head=d_head,
        max_seq_len=64, max_batch_size=1
    )

    # Simulate a 5-token prompt
    prompt_len = 5
    dummy_hidden = torch.randn(1, prompt_len, n_heads * d_head)

    print(f"Prefilling with prompt of length {prompt_len}...")
    cache, logits = prefill_phase(hidden_states=dummy_hidden, cache=cache)

    print(f"Logits shape after prefill: {tuple(logits.shape)}")
    print(f"  [batch=1, seq_len={prompt_len}, d_model={n_heads * d_head}]")
    print()
    print("Cache state after prefill (layer 0):")
    if isinstance(cache[0], (list, tuple)):
        k, v = cache[0]
        # Count filled positions
        filled = (k.abs().sum(dim=-1) > 0).sum().item()
        print(f"  K shape: {tuple(k.shape)}, filled positions: {filled}")
        print(f"  V shape: {tuple(v.shape)}")
else:
    print("Complete problems 018 and 022 to run the prefill phase.")

## Cell 4: Run decode phase step by step, show cache growing

In [ ]:
import importlib
import torch

try:
    _m = importlib.import_module("solutions.023_decode_phase")
    decode_phase = _m.decode_phase
except Exception:
    print("Solve problem 023 first:")
    print("  cp problems/023_decode_phase.py solutions/023_decode_phase.py")
    decode_phase = None

try:
    _m = importlib.import_module("solutions.018_allocate_kv_cache_buffers")
    allocate_kv_cache_buffers = _m.allocate_kv_cache_buffers
except Exception:
    allocate_kv_cache_buffers = None

if decode_phase is not None and allocate_kv_cache_buffers is not None:
    n_layers, n_heads, d_head = 2, 4, 16
    max_seq = 20
    cache = allocate_kv_cache_buffers(
        n_layers=n_layers, n_heads=n_heads, d_head=d_head,
        max_seq_len=max_seq, max_batch_size=1
    )

    # Simulate a previously prefilled prompt of length 3
    prompt_len = 3
    cache_len = prompt_len  # pretend we already filled 3 positions

    print("Decode step-by-step:")
    print(f"  Starting from prompt_len={prompt_len}")
    print()

    for step in range(5):
        # One token input
        token_hidden = torch.randn(1, 1, n_heads * d_head)
        cache, next_logits, cache_len = decode_phase(
            hidden_states=token_hidden,
            cache=cache,
            cache_len=cache_len,
        )
        next_token = next_logits.argmax(dim=-1).item()
        print(f"  Step {step+1}: cache_len={cache_len}, next_token_id={next_token}")
else:
    print("Complete problems 018 and 023 to run the decode phase.")

## Cell 5: Benchmark — with vs without KV cache

In [ ]:
import importlib
import time
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    _m = importlib.import_module("solutions.024_benchmark_kv_cache_speedup")
    benchmark_kv_cache_speedup = _m.benchmark_kv_cache_speedup
except Exception:
    print("Solve problem 024 first:")
    print("  cp problems/024_benchmark_kv_cache_speedup.py solutions/024_benchmark_kv_cache_speedup.py")
    benchmark_kv_cache_speedup = None

if benchmark_kv_cache_speedup is not None:
    results = benchmark_kv_cache_speedup()
    print("Benchmark results:")
    for r in results:
        print(f"  seq_len={r['seq_len']:>4}: "
              f"no_cache={r['time_no_cache']:.4f}s  "
              f"with_cache={r['time_with_cache']:.4f}s  "
              f"speedup={r['speedup']:.1f}x")
    seq_lens = [r['seq_len'] for r in results]
    speedups = [r['speedup'] for r in results]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(seq_lens, speedups, 'o-', color="steelblue")
    ax.set_xlabel("Sequence Length")
    ax.set_ylabel("Speedup (x)")
    ax.set_title("KV Cache Speedup vs Sequence Length")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/tmp/kv_cache_speedup.png", dpi=100)
    plt.show()
else:
    # Show what the benchmark should look like
    seq_lens = [10, 25, 50, 100, 200]
    # Theoretical speedup: sum(i^2) / sum(i) = (2n+1)/3
    speedups = [(2*n+1)/3 for n in seq_lens]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(seq_lens, speedups, 'o--', color="gray")
    ax.set_xlabel("Sequence Length")
    ax.set_ylabel("Speedup (x)")
    ax.set_title("Expected KV Cache Speedup (theoretical)\n[solve problem 024 for actual benchmark]")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()